# Shared Functions

Helper functions used by the bronze, silver and gold notebooks. Other notebooks load them with `%run NB_FUNCTIONS`.

| Function | What it does |
|---|---|
| `add_metadata_ingestion` | Adds `_source_file`, `_source`, `_loaded_at` and `_batch_id` columns |
| `corrupted_record` | Selects rows with a corrupt-record value, shaped for the quarantine folder |
| `write_application_logs` | Appends one row of clean and corrupt record counts to the audit Delta table |
| `normalise_null` | Turns null-like strings (`null`, empty, `n/a`, `na`, `-`, `none`) into real nulls |
| `mapping_values` | Maps a column's values through a dictionary (keys matched trimmed and upper-cased) |
| `date_parse` | Parses dates written in several formats |
| `amount_parse` | Parses currency-like strings to decimal; values in parentheses become negative |
| `interest_rate_parse` | Normalises interest rates to percentages (handles `%` and fractions) |
| `parse_txn_ts` | Parses epoch-millis and ISO or naive timestamps; naive values are assumed to be IST (+05:30) |
| `flatten` | Flattens nested struct columns into `parent_child` columns |
| `latest_by_key` | Keeps the most recent row per key |
| `apply_rules` | Evaluates SQL-expression rules into a `dq_issues` array |
| `routing` | Quarantines rows with `ERROR` issues to Parquet and upserts the rest into a Delta table |
| `dq_summary` | Counts failed rows per rule |
| `delta_merge` | Creates a table on the first run, then merges on a key |


## Imports

In [ ]:
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
)

from delta.tables import DeltaTable
from datetime import datetime
import logging

## Ingestion and audit helpers

In [ ]:
# Ingesting Metadata
def add_metadata_ingestion(df, source_file, source_name, batch_id):
    return (
        df.withColumn("_source_file", F.lit(source_file))
        .withColumn("_source", F.lit(source_name))
        .withColumn("_loaded_at", F.current_timestamp())
        .withColumn("_batch_id", F.lit(batch_id))
    )

In [ ]:
# Quarantine Corrupted Record
def corrupted_record(df, entity_name):
    return df.filter(F.col("_corrupted_record").isNotNull()).select(
        F.lit(entity_name).alias("entity"),
        F.col("_batch_id"),
        F.col("_source_file"),
        F.col("_source"),
        F.col("_corrupted_record"),
        F.current_timestamp().alias("_quarantined_at"),
    )

In [ ]:
# Logging as delta table
def write_application_logs(
    logger_name: str,
    df_name: str,
    layer: str,
    no_of_corr_rec: int,
    no_of_cln_rec: int,
    log_save_path: str,
    cln_path: str,
    corr_path: str,
    batch_id: str,
):
    try:
        spark = SparkSession.builder.getOrCreate()

        schema = StructType(
            [
                StructField("timestamp", TimestampType(), True),
                StructField("logger_name", StringType(), True),
                StructField("batch_id", StringType(), True),
                StructField("layer", StringType(), True),
                StructField("dataframe_name", StringType(), True),
                StructField("status", StringType(), True),
                StructField("clean_records_count", LongType(), True),
                StructField("corrupt_records_count", LongType(), True),
                StructField("clean_output_path", StringType(), True),
                StructField("corrupt_output_path", StringType(), True),
            ]
        )
        status = "WARNING: Corrupt Records Found" if no_of_corr_rec > 0 else "SUCCESS"

        log_data = [
            (
                datetime.now(),
                logger_name,
                batch_id,
                layer,
                df_name,
                status,
                int(no_of_cln_rec),
                int(no_of_corr_rec),
                cln_path,
                corr_path,
            )
        ]

        log_df = spark.createDataFrame(log_data, schema)

        log_df.write.format("delta").mode("append").saveAsTable(log_save_path)

        print(
            f"Log successfully written to Delta table 'etl_audit_logs' for DataFrame: '{df_name}' [Batch: {batch_id}]"
        )
        if no_of_corr_rec > 0:
            print(f"ALERT: {no_of_corr_rec} corrupt records quarantined to {corr_path}")

    except Exception as e:
        print(
            f"CRITICAL ERROR 502: Failed to save log row to Delta table. Details: {str(e)}"
        )

## Cleaning and transformation helpers

In [ ]:
# Normalise Nulls Function
NULL_TOKENS = ["null", "", "n/a", "na", "-", "none"]


def normalise_null(df):
    cols = []
    for c, t in df.dtypes:
        if t == "string" and not c.startswith("_"):
            v = F.trim(F.col(c))
            cols.append(
                F.when(F.lower(v).isin(NULL_TOKENS), None).otherwise(F.col(c)).alias(c)
            )
        else:
            cols.append(F.col(c))
    return df.select(*cols)

In [ ]:
# Mapping Values Function
def mapping_values(col, mapping, default=None):
    m = F.create_map(*[F.lit(x) for kv in mapping.items() for x in kv])
    v = m[F.upper(F.trim(F.col(col)))]
    return F.coalesce(v, F.lit(default)) if default is not None else v

In [ ]:
# Date Parsing Function
def date_parse(
    col, date_format=["yyyy-MM-dd", "dd/MM/yyyy", "dd-MMM-yyyy", "yyyy/MM/dd"]
):
    s = F.trim(F.col(col))
    return F.coalesce(*[F.to_date(s, f) for f in date_format])

In [ ]:
# Amount Parsing Function
def amount_parse(col, scale="DECIMAL(18,2)"):
    s = F.trim(F.col(col))
    negative = s.rlike(r"^\(.*\)$")
    digit = F.regexp_replace(s, r"[^0-9.\-]", "")
    value = F.when(digit.rlike(r"^-?\d+(\.\d+)?$"), digit.cast(scale))
    return F.when(negative, -value).otherwise(value)

In [ ]:
# Latest By Key
def latest_by_key(df, key_col, order_col):
    window = Window.partitionBy(*key_col).orderBy(
        *[F.col(c).desc_nulls_last() for c in order_col]
    )
    return (
        df.withColumn("_rn", F.row_number().over(window)).filter("_rn==1").drop("_rn")
    )

In [ ]:
# Parsing interest_rate
def interest_rate_parse(col):
    s = F.trim(F.col(col))
    v = F.regexp_replace(s, "%", "").cast("double")
    return F.when(s.endswith("%") | (v >= 1), v).otherwise(F.round(v * 100, 4))

In [ ]:
# Normalise Timestamp to UTC
def parse_txn_ts(col):
    s = F.trim(F.col(col))
    return F.coalesce(
        F.when(
            s.rlike(r"^\d{13}$"),
            F.expr(f"timestamp_millis(cast(trim({col}) AS BIGINT))"),
        ),
        F.to_timestamp(s, "yyyy-MM-dd'T'HH:mm:ssXXX"),  # ISO + offset
        F.to_timestamp(
            F.concat(s, F.lit("+05:30")), "yyyy-MM-dd HH:mm:ssXXX"
        ),  # naive IST
        F.to_timestamp(F.concat(s, F.lit("+05:30")), "dd/MM/yyyy HH:mmXXX"),
    )

In [ ]:
# Flatten Nested
def flatten(df, sep="_"):

    while True:  # exits when no struct columns are left
        structs = [
            f.name for f in df.schema.fields if isinstance(f.dataType, StructType)
        ]

        if not structs:
            return df

        cols = []

        for f in df.schema.fields:
            if f.name in structs:
                cols += [
                    F.col(f"`{f.name}`.`{s.name}`").alias(f"{f.name}{sep}{s.name}")
                    for s in f.dataType.fields
                ]
            else:
                cols.append(F.col(f"`{f.name}`"))

        df = df.select(*cols)

## Data quality functions

In [ ]:
# Ingesting DQ results
def apply_rules(df, rules):
    checks = [
        F.when(
            ~F.coalesce(F.expr(r["expr"]), F.lit(False)),
            F.lit(f"{r['severity']}:{r['name']}"),
        )
        for r in rules
    ]

    return df.withColumn(
        "dq_issues", F.filter(F.array(*checks), lambda x: x.isNotNull())
    )

In [ ]:
# Routing Data
def routing(df, entity, schema, table_name, merge_condition, match_condition):

    has_error = F.exists(F.col("dq_issues"), lambda x: x.startswith("ERROR"))

    good = df.filter(~has_error)

    bad = df.filter(has_error).select(
        F.lit(entity).alias("entity"),
        F.array_join("dq_issues", "; ").alias("reason"),
        F.to_json(F.struct(*[F.col(c) for c in df.columns])).alias("payload"),
        F.current_timestamp().alias("quarantined_at"),
    )

    # Route invalid records to quarantine
    bad.write.mode("append").parquet(f"Files/quarantine/DQ_Rejection/{entity}")

    # Construct fully qualified table name
    full_table_name = f"{schema}.{table_name}"

    # Route valid records to Silver
    if not spark.catalog.tableExists(full_table_name):

        # First batch: Create Silver table
        good.write.format("delta").mode("overwrite").saveAsTable(full_table_name)

    else:
        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

        # Subsequent batches: MERGE
        silver_table = DeltaTable.forName(spark, full_table_name)

        (
            silver_table.alias("target")
            .merge(good.alias("source"), merge_condition)
            .whenMatchedUpdateAll(condition=match_condition)
            .whenNotMatchedInsertAll()
            .execute()
        )

    return good

In [ ]:
# DQ Summary Report
def dq_summary(df, entity, batch_id):
    total = df.count()
    return (
        df.select(F.explode("dq_issues").alias("issue"))
        .groupBy("issue")
        .count()
        .select(
            F.lit(batch_id).alias("batch_id"),
            F.lit(entity).alias("entity"),
            F.split("issue", ":")[0].alias("severity"),
            F.split("issue", ":")[1].alias("rule"),
            F.col("count").alias("failed_rows"),
            F.round(F.col("count") / total * 100, 2).alias("failed_pct"),
        )
    )

## Loading

In [ ]:
# Delta Table Merge
from delta.tables import DeltaTable


def delta_merge(df, schema, table_name, merge_cond, match_cond=None):

    full_table_name = f"{schema}.{table_name}"

    if not spark.catalog.tableExists(full_table_name):
        sql_statement = f"CREATE SCHEMA IF NOT EXISTS {schema}"
        spark.sql(sql_statement)
        df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)

    else:
        gold_table = DeltaTable.forName(spark, full_table_name)

        (
            gold_table.alias("target")
            .merge(df.alias("source"), merge_cond)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    return "SUCCESSFULLY DONE"